# mplstudio × scanpy

mplstudio detects the two most common scanpy plot styles and adds a **`palette=`** kwarg to carry scanpy's colour assignments directly into the GUI.

| Pattern | What scanpy does | What mplstudio does |
|---|---|---|
| Categorical UMAP | One `PathCollection` per category, no `ax.legend()` | Detects unlabeled collections, falls back to direct scan |
| Continuous UMAP | Single `PathCollection` + `set_array()`, label `""` | Picks up scalar-mapped collection, shows colormap controls |

**Install:**
```bash
pip install mplstudio scanpy
```

The notebook runs fine without scanpy too — it falls back to synthetic data that reproduces the same matplotlib patterns.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mplstudio

try:
    import scanpy as sc
    HAS_SCANPY = True
    print(f"scanpy {sc.__version__} found — using real AnnData")
except ImportError:
    HAS_SCANPY = False
    print("scanpy not installed — using synthetic data that mirrors scanpy's matplotlib output")

## 1. Categorical UMAP

Scanpy draws one `PathCollection` per category and no `ax.legend()` (it renders its own external legend via `AnchoredOffsetbox`).  
mplstudio's fallback scan picks up all labeled collections automatically.

In [ ]:
# Palette that mirrors what would live in adata.uns["cell_type_colors"]
CELL_TYPE_COLORS = {
    "T cell":    "#e41a1c",
    "B cell":    "#377eb8",
    "NK cell":   "#4daf4a",
    "Monocyte":  "#ff7f00",
    "DC":        "#984ea3",
}

if HAS_SCANPY:
    # ── real scanpy path ──────────────────────────────────────────────────
    adata = sc.datasets.pbmc68k_reduced()
    sc.pp.neighbors(adata)
    sc.tl.umap(adata)

    fig_cat, ax_cat = plt.subplots(figsize=(5, 4))
    sc.pl.umap(adata, color="bulk_labels", ax=ax_cat, show=False)
    # adata.uns["bulk_labels_colors"] holds the per-category hex strings
    palette = dict(zip(
        adata.obs["bulk_labels"].cat.categories,
        adata.uns["bulk_labels_colors"],
    ))
    plt.close()

else:
    # ── synthetic fallback: same matplotlib structure as scanpy ───────────
    rng = np.random.default_rng(0)

    # Simulate UMAP 2-D embedding per cell type
    centres = {"T cell": (2, 1), "B cell": (-2, 1), "NK cell": (0, -2),
               "Monocyte": (2, -1), "DC": (-2, -1)}

    fig_cat, ax_cat = plt.subplots(figsize=(5, 4))
    for ct, (cx, cy) in centres.items():
        xy = rng.standard_normal((80, 2)) * 0.6 + [cx, cy]
        # scanpy: one PathCollection per category, label=category name
        ax_cat.scatter(xy[:, 0], xy[:, 1],
                       c=CELL_TYPE_COLORS[ct], label=ct, s=8, alpha=0.85)
    # scanpy does NOT call ax.legend() here
    ax_cat.set_xlabel("UMAP1")
    ax_cat.set_ylabel("UMAP2")
    ax_cat.set_title("UMAP — cell type (synthetic)")
    plt.close()

    palette = CELL_TYPE_COLORS.copy()

print("Palette:", palette)

In [ ]:
# Pass the palette so mplstudio shows a "Custom" colour mode
# that pre-fills each category with its original scanpy colour.
mplstudio.studio(fig_cat, palette=palette)

**What to try in the panel:**

- **Colors → Custom**: restores the original scanpy palette at any time.
- **Colors → Manual**: individual pickers are pre-filled from the palette — tweak single categories.
- **Colors → Palette / Smart**: switch to a built-in palette or a perceptually-uniform auto palette.
- **Typography**: adjust font size and family for axis labels / title.
- **Axes**: tighten axis limits or add/remove grid.

## 2. Continuous UMAP (gene expression)

Scanpy plots gene expression as a single `PathCollection` with `set_array()` (per-cell values) and label `""`.  
mplstudio detects this as a *continuous* plot and shows the colormap picker.

In [ ]:
if HAS_SCANPY:
    fig_cont, ax_cont = plt.subplots(figsize=(5, 4))
    gene = adata.var_names[0]  # first gene
    sc.pl.umap(adata, color=gene, ax=ax_cont, show=False)
    plt.close()

else:
    rng2 = np.random.default_rng(1)
    n = 400
    angle = rng2.uniform(0, 2 * np.pi, n)
    radius = rng2.uniform(0, 3, n)
    x = radius * np.cos(angle)
    y = radius * np.sin(angle)
    # Gene expression: higher near the centre
    expr = np.exp(-radius / 1.5) + rng2.standard_normal(n) * 0.05
    expr = np.clip(expr, 0, None)

    fig_cont, ax_cont = plt.subplots(figsize=(5, 4))
    # scanpy: single scatter, label="", c=array via set_array internally
    scat = ax_cont.scatter(x, y, c=expr, cmap="viridis", s=8, vmin=0)
    fig_cont.colorbar(scat, ax=ax_cont, label="expr (log-norm)")
    ax_cont.set_xlabel("UMAP1")
    ax_cont.set_ylabel("UMAP2")
    ax_cont.set_title("UMAP — gene expression (synthetic)")
    plt.close()

mplstudio.studio(fig_cont)

**What to try:**

- **Colors → Sequential**: swap `viridis` for `plasma`, `inferno`, or `magma`.
- **Colors → Diverging**: try `RdBu_r` or `coolwarm` to highlight cells above / below the mean.
- **Save**: export a publication-ready PNG at 300 dpi.

## 3. Heatmap (`sc.pl.heatmap`)

`sc.pl.heatmap` renders a gene × group matrix as `imshow` — a continuous artist.  
mplstudio detects it automatically and shows the **colormap picker** with a gradient preview.

In [ ]:
MARKER_GENES = {
    "T cell":   ["CD3D", "CD3E", "CD8A"],
    "B cell":   ["CD79A", "MS4A1", "CD19"],
    "NK cell":  ["GNLY", "NKG7", "KLRD1"],
    "Monocyte": ["LYZ", "CST3", "CD14"],
    "DC":       ["FCER1A", "CST7", "CLEC9A"],
}
CELL_TYPES = list(MARKER_GENES)
GENES      = [g for gs in MARKER_GENES.values() for g in gs]

if HAS_SCANPY:
    # sc.pl.heatmap returns a dict of axes; the main heatmap axes contains an AxesImage
    axes_dict = sc.pl.heatmap(
        adata,
        var_names=adata.var_names[:12],
        groupby="bulk_labels",
        show=False,
    )
    fig_hm = axes_dict["heatmap_ax"].get_figure()
    plt.close()

else:
    # Synthetic: mean log-normalised expression per cell type per gene
    rng5 = np.random.default_rng(5)
    # Base expression matrix: rows = cell types, cols = genes
    base = np.zeros((len(CELL_TYPES), len(GENES)))
    for i, ct in enumerate(CELL_TYPES):
        for j, gene in enumerate(GENES):
            # high if gene is a marker for this cell type
            base[i, j] = 2.5 if gene in MARKER_GENES[ct] else 0.2
    expr_matrix = (base + rng5.exponential(0.3, base.shape)).T  # genes × cell_types

    fig_hm, ax_hm = plt.subplots(figsize=(6, 5))
    im = ax_hm.imshow(expr_matrix, aspect="auto", cmap="Reds", vmin=0)
    fig_hm.colorbar(im, ax=ax_hm, label="Mean expression (log-norm)")
    ax_hm.set_xticks(range(len(CELL_TYPES)))
    ax_hm.set_xticklabels(CELL_TYPES, rotation=35, ha="right", fontsize=9)
    ax_hm.set_yticks(range(len(GENES)))
    ax_hm.set_yticklabels(GENES, fontsize=8)
    ax_hm.set_title("Marker gene expression per cell type")
    plt.tight_layout()
    plt.close()

mplstudio.studio(fig_hm)

**What to try:**

- **Colors → Sequential**: `Reds` → `YlOrRd`, `viridis`, `plasma` 등으로 바꿔보기.
- **Colors → Diverging**: `RdBu_r`나 `coolwarm`으로 up/down-regulated 유전자 강조.
- **Typography**: 폰트 크기를 키우면 유전자 이름과 cell type 레이블 가독성이 올라감.
- **Figure size**: 유전자 수에 맞게 세로 길이 조정.

## 4. Violin plot (`sc.pl.violin`)

`sc.pl.violin`은 cell type별 유전자 발현 분포를 보여주는 그래프.  
Violin 패치(`PolyCollection`)는 레이블이 없기 때문에 **Colors 섹션은 "No labeled series"** 로 표시되며 색 변경은 지원되지 않는다.  
대신 **Typography, Figure size, Grid, Spines** 컨트롤이 논문용 다듬기에 유용하다.

In [ ]:
VIOLIN_GENES = ["CD3D", "CD79A", "LYZ"]  # T cell, B cell, Monocyte markers

if HAS_SCANPY:
    fig_vn, axes_vn = plt.subplots(1, len(VIOLIN_GENES), figsize=(9, 3), sharey=False)
    for ax_v, gene in zip(axes_vn, VIOLIN_GENES):
        sc.pl.violin(adata, keys=gene, groupby="bulk_labels",
                     ax=ax_v, show=False, rotation=35)
    fig_vn.suptitle("Marker gene expression by cell type")
    plt.tight_layout()
    plt.close()

else:
    rng6 = np.random.default_rng(6)

    # Simulate per-cell expression for 3 marker genes across 5 cell types
    # Each gene is highly expressed in its cognate cell type
    COGNATE = {"CD3D": "T cell", "CD79A": "B cell", "LYZ": "Monocyte"}
    N_CELLS = 120  # per cell type

    fig_vn, axes_vn = plt.subplots(1, len(VIOLIN_GENES), figsize=(9, 3), sharey=False)
    for ax_v, gene in zip(axes_vn, VIOLIN_GENES):
        all_data, all_pos, tick_labels = [], [], []
        for k, ct in enumerate(CELL_TYPES):
            high = (ct == COGNATE[gene])
            mu  = rng6.uniform(1.8, 2.5) if high else rng6.uniform(0.0, 0.4)
            sig = 0.4 if high else 0.3
            vals = np.clip(rng6.normal(mu, sig, N_CELLS), 0, None)
            all_data.append(vals)
            tick_labels.append(ct)

        # violinplot + strip-chart (same structure as scanpy)
        parts = ax_v.violinplot(all_data, positions=range(len(CELL_TYPES)),
                                showmedians=True, showextrema=False, widths=0.7)
        for k, vals in enumerate(all_data):
            jitter = rng6.uniform(-0.12, 0.12, len(vals))
            ax_v.scatter(k + jitter, vals, s=2, alpha=0.3,
                         color="#555555", zorder=3)
        # apply palette colours to violin bodies
        for body, ct in zip(parts["bodies"], CELL_TYPES):
            body.set_facecolor(CELL_TYPE_COLORS[ct])
            body.set_alpha(0.75)

        ax_v.set_title(gene, fontsize=10)
        ax_v.set_xticks(range(len(CELL_TYPES)))
        ax_v.set_xticklabels(tick_labels, rotation=35, ha="right", fontsize=8)
        ax_v.set_ylabel("Expression" if ax_v is axes_vn[0] else "")

    fig_vn.suptitle("Marker gene expression by cell type")
    plt.tight_layout()
    plt.close()

mplstudio.studio(fig_vn)

**What to try:**

- **Typography**: 폰트 크기를 높이면 유전자 이름(subplot title)과 cell type x-tick 레이블 가독성이 올라감.
- **Grid → On**: 수평 grid line을 켜면 발현 수치를 읽기 쉬워짐.
- **Spines → 2-Side** 또는 **None**: 논문 스타일에 맞게 spine을 정리.
- **Figure size**: 유전자를 더 추가할 때 가로 길이를 늘려 서브플롯 간격 확보.